# 简单线性回归完整教程：从最小二乘法到模型诊断

## 📚 教学目标
1. 理解最小二乘法的几何原理
2. 掌握回归方程 $\hat{y} = b_0 + b_1 x$ 的完整计算步骤
3. 理解残差和残差分析的方法
4. 识别强影响点及其对回归的影响
5. 评估回归模型的拟合优度（决定系数 $r^2$）

**参考书**：《基础统计学(第14版)》（Triola）第 10-2 节
**教学策略**：先用极小数据集（n=10）让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：用学习时间预测考试成绩

### 🎯 问题

一位教师想知道：**学生的学习时间能否预测考试成绩？**

我们收集了学生每周的学习时间（小时）和对应的考试成绩，想找到一条"最佳拟合直线"来描述两者的关系。

### 📖 书中的定义

> 回归分析（regression analysis）是研究两个变量之间关系的一种统计方法。
> 如果存在显著的线性关系，我们可以用回归方程来描述这种关系，并用它进行预测。

### 📐 回归方程

$$\hat{y} = b_0 + b_1 x$$

其中：
- $\hat{y}$ = 因变量的预测值（考试成绩）
- $b_0$ = 截距（y 轴截距，x=0 时的预测值）
- $b_1$ = 斜率（x 每增加 1 单位，y 的变化量）
- $x$ = 自变量（学习时间）

### 📐 最小二乘法的核心思想

最小二乘法（Least Squares Method）选择使得**残差平方和**最小的直线：

$$\sum (y - \hat{y})^2 = \sum (y - b_0 - b_1 x)^2 \rightarrow \text{最小}$$

这保证了直线在"总体上"离所有数据点最近。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 最小二乘法的几何直觉

### 💡 为什么叫"最小二乘"？

想象在散点图上画一条直线。对于每个数据点 $(x_i, y_i)$：
- **残差** $e_i = y_i - \hat{y}_i$ 是点到直线的垂直距离
- 最小二乘法选择使得 $\sum e_i^2$（残差平方和）最小的直线

### 📐 斜率和截距的计算公式

$$b_1 = \frac{n \sum xy - (\sum x)(\sum y)}{n \sum x^2 - (\sum x)^2}$$

$$b_0 = \bar{y} - b_1 \bar{x}$$

其中：
- $n$ = 样本量
- $\bar{x}$ = x 的样本均值
- $\bar{y}$ = y 的样本均值

### 📐 斜率的另一种形式（利用相关系数）

$$b_1 = r \cdot \frac{s_y}{s_x}$$

其中：
- $r$ = 相关系数
- $s_x, s_y$ = x 和 y 的样本标准差

In [ ]:
# ========== 最小二乘法几何直觉可视化 ==========

# 生成一组简单数据演示
x_demo = np.array([1, 2, 3, 4, 5, 6, 7])
y_demo = np.array([2.1, 3.9, 6.2, 7.8, 10.1, 12.3, 13.8])

# 用最小二乘法拟合
slope_demo, intercept_demo, _, _, _ = stats.linregress(x_demo, y_demo)
y_pred_demo = intercept_demo + slope_demo * x_demo
residuals_demo = y_demo - y_pred_demo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：散点 + 回归线 + 残差 ---
ax1 = axes[0]
ax1.scatter(x_demo, y_demo, color='steelblue', s=80, zorder=5, label='Data Points')
ax1.plot(x_demo, y_pred_demo, color='#e74c3c', linewidth=2, label=f'Regression Line')

for i in range(len(x_demo)):
    ax1.plot([x_demo[i], x_demo[i]], [y_demo[i], y_pred_demo[i]],
             color='#2ecc71', linewidth=1.5, linestyle='--', alpha=0.7)

ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.set_title('Least Squares: Minimize Vertical Distances', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 右图：残差柱状图 ---
ax2 = axes[1]
colors_res = ['#2ecc71' if r >= 0 else '#e74c3c' for r in residuals_demo]
ax2.bar(x_demo, residuals_demo, color=colors_res, edgecolor='white', alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('Residual (y - ŷ)', fontsize=12)
ax2.set_title('Residuals: Distance from Line', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：绿色虚线表示每个数据点到回归线的垂直距离（残差）")
print("  右图：正残差（绿色）表示点在线上方，负残差（红色）表示点在线下方")
print("  最小二乘法的目标就是让这些残差的平方和最小")

---

## 3. 微型数据手算：回归方程

### 📊 数据

我们用 10 名学生的学习时间和考试成绩数据：

| 学生 | 学习时间 x | 考试成绩 y |
|------|-----------|----------|
| 1    | 2         | 50       |
| 2    | 3         | 55       |
| 3    | 4         | 65       |
| 4    | 5         | 70       |
| 5    | 6         | 74       |
| 6    | 7         | 80       |
| 7    | 8         | 85       |
| 8    | 9         | 88       |
| 9    | 10        | 92       |
| 10   | 12        | 95       |

In [ ]:
# ========== 步骤 1: 微型数据集 ==========

x_micro = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 12])
y_micro = np.array([50, 55, 65, 70, 74, 80, 85, 88, 92, 95])
n_micro = len(x_micro)

print("📊 步骤 1: 微型数据集")
print(f"  样本量 n = {n_micro}")
print(f"  x (学习时间): {x_micro}")
print(f"  y (考试成绩): {y_micro}")

# ========== 步骤 2: 计算基本统计量 ==========

x_bar = np.mean(x_micro)
y_bar = np.mean(y_micro)

print(f"\n📊 步骤 2: 计算均值")
print(f"  x̄ = {x_bar:.4f}")
print(f"  ȳ = {y_bar:.4f}")

# ========== 步骤 3: 计算中间量 ==========

sum_x = np.sum(x_micro)
sum_y = np.sum(y_micro)
sum_xy = np.sum(x_micro * y_micro)
sum_x2 = np.sum(x_micro ** 2)
sum_y2 = np.sum(y_micro ** 2)

print(f"\n📊 步骤 3: 计算中间量")
print(f"  Σx   = {sum_x}")
print(f"  Σy   = {sum_y}")
print(f"  Σxy  = {sum_xy}")
print(f"  Σx²  = {sum_x2}")
print(f"  Σy²  = {sum_y2}")

In [ ]:
# ========== 步骤 4: 计算斜率 b₁ ==========

b1_numerator = n_micro * sum_xy - sum_x * sum_y
b1_denominator = n_micro * sum_x2 - sum_x ** 2
b1 = b1_numerator / b1_denominator

print(f"📊 步骤 4: 计算斜率 b₁")
print(f"  公式: b₁ = [nΣxy - (Σx)(Σy)] / [nΣx² - (Σx)²]")
print(f"  分子 = {n_micro}×{sum_xy} - {sum_x}×{sum_y} = {b1_numerator}")
print(f"  分母 = {n_micro}×{sum_x2} - {sum_x}² = {b1_denominator}")
print(f"  b₁ = {b1_numerator} / {b1_denominator} = {b1:.4f}")
print(f"\n  💡 解读: 学习时间每增加 1 小时，考试成绩平均增加 {b1:.2f} 分")

# ========== 步骤 5: 计算截距 b₀ ==========

b0 = y_bar - b1 * x_bar

print(f"\n📊 步骤 5: 计算截距 b₀")
print(f"  公式: b₀ = ȳ - b₁x̄")
print(f"  b₀ = {y_bar:.4f} - {b1:.4f}×{x_bar:.4f}")
print(f"  b₀ = {y_bar:.4f} - {b1 * x_bar:.4f} = {b0:.4f}")
print(f"\n  💡 解读: 当学习时间为 0 小时，预测成绩为 {b0:.2f} 分")

# ========== 步骤 6: 写出回归方程 ==========

print(f"\n📊 步骤 6: 回归方程")
print(f"  ŷ = {b0:.4f} + {b1:.4f}x")
print(f"  即: 预测成绩 = {b0:.2f} + {b1:.2f} × 学习时间")

### 📐 用相关系数验证斜率

另一种计算斜率的方法：$b_1 = r \cdot \dfrac{s_y}{s_x}$

In [ ]:
# ========== 步骤 7: 用相关系数验证斜率 ==========

r_manual, _ = stats.pearsonr(x_micro, y_micro)
sx = np.std(x_micro, ddof=1)
sy = np.std(y_micro, ddof=1)
b1_from_r = r_manual * (sy / sx)

print(f"📊 步骤 7: 用相关系数验证斜率")
print(f"  r  = {r_manual:.6f}")
print(f"  sx = {sx:.4f}")
print(f"  sy = {sy:.4f}")
print(f"  b₁ = r × (sy/sx) = {r_manual:.6f} × ({sy:.4f}/{sx:.4f}) = {b1_from_r:.4f}")
print(f"  手算 b₁ = {b1:.4f}")
print(f"  差异 = {abs(b1 - b1_from_r):.10f}")
print(f"\n  ✅ 两种方法结果一致！")

---

## 4. 用 scipy.stats.linregress 验证

### 🔬 scipy 验证

`scipy.stats.linregress` 函数一次性计算回归方程的所有参数。

In [ ]:
# ========== 用 scipy 验证 ==========

result = stats.linregress(x_micro, y_micro)
b1_scipy = result.slope
b0_scipy = result.intercept
r_scipy = result.rvalue
p_scipy = result.pvalue
se_scipy = result.stderr

print("🔬 scipy.stats.linregress 验证:")
print(f"  手算 斜率 b₁    = {b1:.6f}")
print(f"  scipy 斜率 b₁   = {b1_scipy:.6f}")
print(f"  手算 截距 b₀    = {b0:.6f}")
print(f"  scipy 截距 b₀   = {b0_scipy:.6f}")
print(f"  手算 r          = {r_manual:.6f}")
print(f"  scipy r         = {r_scipy:.6f}")
print(f"  scipy p 值       = {p_scipy:.6f}")
print(f"  scipy 斜率标准误 = {se_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

# 计算预测值和残差
y_hat_micro = b0 + b1 * x_micro
residuals_micro = y_micro - y_hat_micro

print(f"\n📊 回归预测结果:")
print(f"  {'x':>4s}  {'y':>4s}  {'ŷ':>8s}  {'残差 e':>8s}")
print(f"  {'-'*4}  {'-'*4}  {'-'*8}  {'-'*8}")
for i in range(n_micro):
    print(f"  {x_micro[i]:4d}  {y_micro[i]:4d}  {y_hat_micro[i]:8.2f}  {residuals_micro[i]:8.2f}")
print(f"\n  残差之和 = {np.sum(residuals_micro):.6f}（应接近 0）")

---

## 5. 回归直线的可视化

### 📊 散点图 + 回归线

回归直线一定经过点 $(\bar{x}, \bar{y})$。

In [ ]:
# ========== 回归直线可视化 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：微型数据集 ---
ax1 = axes[0]
ax1.scatter(x_micro, y_micro, color='steelblue', s=80, zorder=5, label='Observed Data')
x_line = np.linspace(min(x_micro) - 1, max(x_micro) + 1, 100)
y_line = b0 + b1 * x_line
ax1.plot(x_line, y_line, color='#e74c3c', linewidth=2, label=f'Regression Line')
ax1.scatter([x_bar], [y_bar], color='#2ecc71', s=150, marker='*', zorder=6,
            label=f'Mean Point ({x_bar:.1f}, {y_bar:.1f})')
ax1.set_xlabel('Study Hours (x)', fontsize=12)
ax1.set_ylabel('Exam Score (y)', fontsize=12)
ax1.set_title('Simple Linear Regression (n=10)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 右图：残差图 ---
ax2 = axes[1]
ax2.scatter(y_hat_micro, residuals_micro, color='steelblue', s=60)
ax2.axhline(y=0, color='#e74c3c', linewidth=1.5, linestyle='--')
ax2.set_xlabel('Predicted Values (ŷ)', fontsize=12)
ax2.set_ylabel('Residuals (y - ŷ)', fontsize=12)
ax2.set_title('Residual Plot', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图：回归直线经过均值点（绿色星号），这是最小二乘法的性质")
print("  右图：残差图中，点应随机分布在 y=0 两侧，无明显模式")

---

## 6. 残差分析

### 📐 残差的定义

$$e_i = y_i - \hat{y}_i$$

其中：
- $e_i$ = 第 i 个观测的残差
- $y_i$ = 实际观测值
- $\hat{y}_i$ = 回归预测值

### 💡 残差分析的三大检查

1. **随机性**：残差应随机分布在 0 附近，无明显模式
2. **等方差性**：残差的散布范围不应随 x 变化而变化
3. **正态性**：残差应近似服从正态分布

In [ ]:
# ========== 残差分析完整版 ==========

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- 图1：残差 vs 预测值 ---
ax1 = axes[0]
ax1.scatter(y_hat_micro, residuals_micro, color='steelblue', s=60)
ax1.axhline(y=0, color='#e74c3c', linewidth=1.5, linestyle='--')
ax1.set_xlabel('Predicted Values', fontsize=12)
ax1.set_ylabel('Residuals', fontsize=12)
ax1.set_title('Residuals vs Predicted', fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)

# --- 图2：残差 vs x ---
ax2 = axes[1]
ax2.scatter(x_micro, residuals_micro, color='steelblue', s=60)
ax2.axhline(y=0, color='#e74c3c', linewidth=1.5, linestyle='--')
ax2.set_xlabel('Study Hours (x)', fontsize=12)
ax2.set_ylabel('Residuals', fontsize=12)
ax2.set_title('Residuals vs x', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

# --- 图3：残差正态 Q-Q 图 ---
ax3 = axes[2]
stats.probplot(residuals_micro, dist="norm", plot=ax3)
ax3.set_title('Normal Q-Q Plot of Residuals', fontsize=13, fontweight='bold')
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 残差正态性检验
if len(residuals_micro) >= 8:
    shapiro_stat, shapiro_p = stats.shapiro(residuals_micro)
    print(f"\n🔬 Shapiro-Wilk 正态性检验:")
    print(f"  W 统计量 = {shapiro_stat:.4f}")
    print(f"  p 值 = {shapiro_p:.4f}")
    if shapiro_p > 0.05:
        print(f"  💡 p > 0.05，不能拒绝正态性假设")
    else:
        print(f"  💡 p ≤ 0.05，残差可能不服从正态分布")

print(f"\n📊 残差统计:")
print(f"  残差均值 = {np.mean(residuals_micro):.6f}（应接近 0）")
print(f"  残差标准差 = {np.std(residuals_micro, ddof=1):.4f}")
print(f"  最大正残差 = {np.max(residuals_micro):.4f}")
print(f"  最大负残差 = {np.min(residuals_micro):.4f}")

print("\n💡 图解说明：")
print("  图1：残差 vs 预测值 — 点应随机分布在 y=0 两侧")
print("  图2：残差 vs x — 不应有明显的曲线模式")
print("  图3：Q-Q 图 — 点应大致落在对角线上，说明残差近似正态")

---

## 7. 决定系数 $r^2$ 的解读

### 📐 决定系数公式

$$r^2 = \frac{\text{解释的变异}}{\text{总变异}} = \frac{SS_{reg}}{SS_{total}} = 1 - \frac{SS_{res}}{SS_{total}}$$

其中：
- $SS_{total} = \sum(y - \bar{y})^2$ = 总平方和（Total Sum of Squares）
- $SS_{reg} = \sum(\hat{y} - \bar{y})^2$ = 回归平方和（Regression Sum of Squares）
- $SS_{res} = \sum(y - \hat{y})^2$ = 残差平方和（Residual Sum of Squares）

### 💡 含义

$r^2$ 表示因变量的变异中，有多少比例可以由自变量解释。
- $r^2 = 0.85$ 意味着自变量解释了因变量 85% 的变异

In [ ]:
# ========== 决定系数 r² 的计算与解读 ==========

# ========== 步骤 1: 计算三种平方和 ==========

SS_total = np.sum((y_micro - y_bar) ** 2)
SS_reg = np.sum((y_hat_micro - y_bar) ** 2)
SS_res = np.sum((y_micro - y_hat_micro) ** 2)

print(f"📊 步骤 1: 计算平方和")
print(f"  SS_total (总平方和)     = {SS_total:.4f}")
print(f"  SS_reg   (回归平方和)   = {SS_reg:.4f}")
print(f"  SS_res   (残差平方和)   = {SS_res:.4f}")
print(f"  验证: SS_reg + SS_res   = {SS_reg + SS_res:.4f}（应等于 SS_total）")

# ========== 步骤 2: 计算 r² ==========

r_squared = 1 - SS_res / SS_total
r_squared_alt = SS_reg / SS_total

print(f"\n📊 步骤 2: 计算决定系数 r²")
print(f"  方法1: r² = 1 - SS_res/SS_total = 1 - {SS_res:.4f}/{SS_total:.4f} = {r_squared:.4f}")
print(f"  方法2: r² = SS_reg/SS_total = {SS_reg:.4f}/{SS_total:.4f} = {r_squared_alt:.4f}")
print(f"  r = {r_manual:.4f}, r² = {r_manual**2:.4f}")

# ========== 步骤 3: 解读 ==========

print(f"\n💡 解读:")
print(f"  r² = {r_squared:.4f} = {r_squared*100:.1f}%")
print(f"  → 学习时间解释了考试成绩 {r_squared*100:.1f}% 的变异")
print(f"  → 剩余 {100 - r_squared*100:.1f}% 的变异来自其他因素")

# 可视化平方和分解
fig, ax = plt.subplots(figsize=(10, 5))
categories = ['SS_total\n(Total)', 'SS_reg\n(Explained)', 'SS_res\n(Unexplained)']
values = [SS_total, SS_reg, SS_res]
colors = ['steelblue', '#2ecc71', '#e74c3c']

bars = ax.bar(categories, values, color=colors, edgecolor='white', alpha=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Sum of Squares', fontsize=12)
ax.set_title('Decomposition of Total Variation', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  蓝色 = 总变异，绿色 = 回归解释的部分，红色 = 未解释的残差")
print(f"  绿色占比 = {r_squared*100:.1f}%，即 r²")

---

## 8. 预测与外推的注意事项

### 📐 用回归方程预测

给定一个新的 x 值，预测 y：

$$\hat{y}_{new} = b_0 + b_1 \cdot x_{new}$$

### ⚠️ 外推的危险

**外推**（extrapolation）是指用回归方程预测 x 范围之外的 y 值。这非常危险，因为我们不知道线性关系是否在范围外仍然成立。

In [ ]:
# ========== 预测与外推 ==========

# ========== 步骤 1: 内插预测（安全） ==========

x_predict_safe = np.array([3, 5, 7, 9])
y_predict_safe = b0 + b1 * x_predict_safe

print(f"📊 步骤 1: 内插预测（x 在数据范围内）")
print(f"  数据范围: x ∈ [{min(x_micro)}, {max(x_micro)}]")
print(f"  {'x':>4s}  {'预测成绩 ŷ':>10s}")
print(f"  {'-'*4}  {'-'*10}")
for x_val, y_val in zip(x_predict_safe, y_predict_safe):
    print(f"  {x_val:4d}  {y_val:10.2f}")

# ========== 步骤 2: 外推预测（危险） ==========

x_predict_extrap = np.array([0, 1, 15, 20])
y_predict_extrap = b0 + b1 * x_predict_extrap

print(f"\n📊 步骤 2: 外推预测（x 超出数据范围，⚠️ 危险！）")
print(f"  {'x':>4s}  {'预测成绩 ŷ':>10s}  {'安全？':>6s}")
print(f"  {'-'*4}  {'-'*10}  {'-'*6}")
for x_val, y_val in zip(x_predict_extrap, y_predict_extrap):
    safe = "✅" if min(x_micro) <= x_val <= max(x_micro) else "⚠️ 外推"
    print(f"  {x_val:4d}  {y_val:10.2f}  {safe}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_micro, y_micro, color='steelblue', s=80, zorder=5, label='Observed Data')

x_full = np.linspace(-2, 22, 100)
y_full = b0 + b1 * x_full

# 安全区间
x_safe = x_full[(x_full >= min(x_micro)) & (x_full <= max(x_micro))]
y_safe = b0 + b1 * x_safe
ax.plot(x_safe, y_safe, color='#2ecc71', linewidth=2.5, label='Interpolation (Safe)')

# 外推区间
x_extrap_left = x_full[x_full < min(x_micro)]
x_extrap_right = x_full[x_full > max(x_micro)]
ax.plot(x_extrap_left, b0 + b1 * x_extrap_left, color='#e74c3c', linewidth=2,
        linestyle='--', label='Extrapolation (Risky)')
ax.plot(x_extrap_right, b0 + b1 * x_extrap_right, color='#e74c3c', linewidth=2,
        linestyle='--')

# 数据范围标记
ax.axvline(x=min(x_micro), color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=max(x_micro), color='gray', linestyle=':', alpha=0.5)
ax.fill_betweenx([0, 100], min(x_micro), max(x_micro), alpha=0.05, color='steelblue')

ax.set_xlabel('Study Hours (x)', fontsize=12)
ax.set_ylabel('Exam Score (y)', fontsize=12)
ax.set_title('Interpolation vs Extrapolation', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(30, 120)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  绿色实线：数据范围内的内插预测（可靠）")
print("  红色虚线：数据范围外的外推预测（不可靠）")
print("  x=0 时预测成绩为 {:.1f} 分，但这是外推，没有实际意义".format(b0))
print("  x=20 时预测成绩为 {:.1f} 分，远超可能的分数范围".format(b0 + b1 * 20))

---

## 9. 强影响点的识别

### 📐 杠杆值（Leverage）

杠杆值 $h_i$ 衡量第 i 个观测在 x 方向上离均值的距离：

$$h_i = \frac{1}{n} + \frac{(x_i - \bar{x})^2}{\sum(x_j - \bar{x})^2}$$

- 杠杆值高（> 2p/n，简单回归中 p=2）的点是**高杠杆点**
- 高杠杆点在 x 方向上是"异常值"，可能对回归线产生很大影响

### 📐 Cook 距离

Cook 距离综合考虑残差和杠杆值，衡量删除第 i 个观测对回归系数的影响：

$$D_i = \frac{e_i^2}{p \cdot MSE} \cdot \frac{h_i}{(1 - h_i)^2}$$

- $D_i > 1$ 通常被认为是强影响点
- $D_i > 4/n$ 也值得关注

In [ ]:
# ========== 强影响点分析 ==========

# ========== 步骤 1: 计算杠杆值 ==========

x_mean = np.mean(x_micro)
SSx = np.sum((x_micro - x_mean) ** 2)
leverage = 1/n_micro + (x_micro - x_mean)**2 / SSx

print(f"📊 步骤 1: 杠杆值 (Leverage)")
print(f"  杠杆值阈值 = 2p/n = 2×2/{n_micro} = {2*2/n_micro:.2f}")
print(f"  {'x':>4s}  {'杠杆值 h':>10s}  {'高杠杆？':>8s}")
print(f"  {'-'*4}  {'-'*10}  {'-'*8}")
threshold_lev = 2 * 2 / n_micro
for i in range(n_micro):
    flag = "⚠️ 是" if leverage[i] > threshold_lev else ""
    print(f"  {x_micro[i]:4d}  {leverage[i]:10.4f}  {flag}")

# ========== 步骤 2: 计算 Cook 距离 ==========

MSE = SS_res / (n_micro - 2)
p = 2  # 参数个数（截距 + 斜率）
cook_d = (residuals_micro**2 / (p * MSE)) * (leverage / (1 - leverage)**2)

print(f"\n📊 步骤 2: Cook 距离")
print(f"  MSE = {MSE:.4f}")
print(f"  Cook 距离阈值 = 4/n = {4/n_micro:.4f}")
print(f"  {'x':>4s}  {'Cook D':>8s}  {'强影响？':>8s}")
print(f"  {'-'*4}  {'-'*8}  {'-'*8}")
threshold_cook = 4 / n_micro
for i in range(n_micro):
    flag = "⚠️ 是" if cook_d[i] > threshold_cook else ""
    print(f"  {x_micro[i]:4d}  {cook_d[i]:8.4f}  {flag}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 杠杆值图
ax1 = axes[0]
colors_lev = ['#e74c3c' if l > threshold_lev else 'steelblue' for l in leverage]
ax1.bar(range(n_micro), leverage, color=colors_lev, edgecolor='white', alpha=0.8)
ax1.axhline(y=threshold_lev, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Threshold={threshold_lev:.2f}')
ax1.set_xlabel('Observation Index', fontsize=12)
ax1.set_ylabel('Leverage', fontsize=12)
ax1.set_title('Leverage Values', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3, axis='y')

# Cook 距离图
ax2 = axes[1]
colors_cook = ['#e74c3c' if c > threshold_cook else 'steelblue' for c in cook_d]
ax2.bar(range(n_micro), cook_d, color=colors_cook, edgecolor='white', alpha=0.8)
ax2.axhline(y=threshold_cook, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Threshold={threshold_cook:.4f}')
ax2.set_xlabel('Observation Index', fontsize=12)
ax2.set_ylabel("Cook's Distance", fontsize=12)
ax2.set_title("Cook's Distance", fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  杠杆值：红色柱表示高杠杆点（x 方向上的异常值）")
print("  Cook 距离：红色柱表示强影响点（删除后回归线变化大）")
print("  x=12 是本数据中杠杆值最高的点，因为它离 x̄ 最远")

---

## 10. 大样本扩展

### 📊 模拟参数

现在我们将样本量扩展到 n=200，模拟更真实的数据场景。

### 📐 数据生成过程 (DGP)

- 学习时间 x: 正态分布, μ=8, σ=3, 裁剪到 [1, 20]
- 考试成绩 y: y = 30 + 6x + ε, ε ~ N(0, 8)
- 真实斜率 = 6, 真实截距 = 30

In [ ]:
# ========== 大样本数据生成与回归分析 ==========

# --- 数据生成过程 (DGP) ---
n_large = 200
x_large = np.random.normal(8, 3, n_large)
x_large = np.clip(x_large, 1, 20)
noise_large = np.random.normal(0, 8, n_large)
y_large = 30 + 6 * x_large + noise_large

print(f"📊 大样本数据生成")
print(f"  样本量 n = {n_large}")
print(f"  DGP: y = 30 + 6x + ε, ε ~ N(0, 8)")
print(f"  x 范围: [{x_large.min():.2f}, {x_large.max():.2f}]")

# ========== 回归分析 ==========

result_large = stats.linregress(x_large, y_large)
b1_large = result_large.slope
b0_large = result_large.intercept
r_large = result_large.rvalue
r2_large = r_large ** 2
p_large = result_large.pvalue

y_hat_large = b0_large + b1_large * x_large
residuals_large = y_large - y_hat_large

print(f"\n📊 回归结果:")
print(f"  真实方程: y = 30 + 6x")
print(f"  估计方程: ŷ = {b0_large:.4f} + {b1_large:.4f}x")
print(f"  r = {r_large:.4f}")
print(f"  r² = {r2_large:.4f} = {r2_large*100:.1f}%")
print(f"  p 值 = {p_large:.6f}")
print(f"\n  💡 学习时间解释了考试成绩 {r2_large*100:.1f}% 的变异")
print(f"  💡 斜率估计误差: |{b1_large:.4f} - 6| = {abs(b1_large - 6):.4f}")

# ========== 可视化 ==========

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1: 散点图 + 回归线
ax1 = axes[0, 0]
ax1.scatter(x_large, y_large, color='steelblue', alpha=0.4, s=30, label='Data')
x_line_large = np.linspace(x_large.min(), x_large.max(), 100)
y_line_large = b0_large + b1_large * x_line_large
y_true = 30 + 6 * x_line_large
ax1.plot(x_line_large, y_line_large, color='#e74c3c', linewidth=2.5, label='Estimated')
ax1.plot(x_line_large, y_true, color='#2ecc71', linewidth=2, linestyle='--', label='True')
ax1.set_xlabel('Study Hours', fontsize=12)
ax1.set_ylabel('Exam Score', fontsize=12)
ax1.set_title('Regression Line (n=200)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 图2: 残差 vs 预测值
ax2 = axes[0, 1]
ax2.scatter(y_hat_large, residuals_large, color='steelblue', alpha=0.4, s=30)
ax2.axhline(y=0, color='#e74c3c', linewidth=1.5, linestyle='--')
ax2.set_xlabel('Predicted Values', fontsize=12)
ax2.set_ylabel('Residuals', fontsize=12)
ax2.set_title('Residuals vs Predicted', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

# 图3: 残差直方图
ax3 = axes[1, 0]
ax3.hist(residuals_large, bins=25, density=True, color='steelblue', edgecolor='white', alpha=0.7)
x_norm = np.linspace(residuals_large.min(), residuals_large.max(), 100)
y_norm = stats.norm.pdf(x_norm, np.mean(residuals_large), np.std(residuals_large))
ax3.plot(x_norm, y_norm, color='#e74c3c', linewidth=2, label='Normal Fit')
ax3.set_xlabel('Residuals', fontsize=12)
ax3.set_ylabel('Density', fontsize=12)
ax3.set_title('Residual Distribution', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

# 图4: Q-Q 图
ax4 = axes[1, 1]
stats.probplot(residuals_large, dist="norm", plot=ax4)
ax4.set_title('Normal Q-Q Plot', fontsize=14, fontweight='bold')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 正态性检验
shapiro_stat, shapiro_p = stats.shapiro(residuals_large)
print(f"\n🔬 残差正态性检验 (Shapiro-Wilk):")
print(f"  W = {shapiro_stat:.4f}, p = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print(f"  💡 p > 0.05，残差满足正态性假设")
else:
    print(f"  💡 p ≤ 0.05，残差可能不满足正态性")

print("\n💡 图解说明：")
print("  左上：回归线与真实关系非常接近，说明估计准确")
print("  右上：残差随机分布，无明显模式")
print("  左下：残差近似正态分布")
print("  右下：Q-Q 图中点大致在对角线上")

---

## 📌 核心概念回顾

### 📌 最小二乘法 (Least Squares Method)
- **定义**: 选择使残差平方和 $\sum(y - \hat{y})^2$ 最小的直线
- **公式**: $b_1 = \frac{n\sum xy - (\sum x)(\sum y)}{n\sum x^2 - (\sum x)^2}$，$b_0 = \bar{y} - b_1\bar{x}$
- **含义**: 保证回归线在"总体上"离所有数据点最近
- **判断标准**: 无，这是计算方法而非判断标准

### 📌 回归方程 (Regression Equation)
- **定义**: 描述因变量与自变量线性关系的方程
- **公式**: $\hat{y} = b_0 + b_1 x$
- **含义**: 给定 x 值，可以用方程预测 y 的值
- **判断标准**: 斜率 b₁ 的符号决定关系方向（正/负）

### 📌 残差 (Residual)
- **定义**: 实际值与预测值之差
- **公式**: $e = y - \hat{y}$
- **含义**: 衡量回归方程预测的误差
- **判断标准**: 残差应随机分布，均值为 0

### 📌 决定系数 (Coefficient of Determination, r²)
- **定义**: 因变量变异中被自变量解释的比例
- **公式**: $r^2 = 1 - \frac{SS_{res}}{SS_{total}}$
- **含义**: r² 越接近 1，回归模型拟合越好
- **判断标准**: r² > 0.7 通常表示强线性关系

### 📌 杠杆值与 Cook 距离 (Leverage & Cook's Distance)
- **定义**: 杠杆值衡量 x 方向的异常程度，Cook 距离衡量删除某点对回归的影响
- **公式**: $h_i = \frac{1}{n} + \frac{(x_i - \bar{x})^2}{\sum(x_j - \bar{x})^2}$
- **含义**: 识别对回归结果有不成比例影响的数据点
- **判断标准**: Cook D > 1 或 Cook D > 4/n

### 🔗 完整流程
```
数据收集 → 绘制散点图 → 计算回归方程 → 残差分析 → r² 评估 → 预测
    ↓            ↓              ↓             ↓          ↓         ↓
  收集(x,y)   检查线性      手算+scipy     检查假设    拟合优度  注意外推
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 | 判断标准 |
|------|------|------|----------|
| 斜率 b₁ | $\frac{n\sum xy - (\sum x)(\sum y)}{n\sum x^2 - (\sum x)^2}$ | x 每增加 1 单位，y 的变化 | 显著不为 0 |
| 截距 b₀ | $\bar{y} - b_1\bar{x}$ | x=0 时 y 的预测值 | 可能无实际意义 |
| 相关系数 r | $\frac{n\sum xy - (\sum x)(\sum y)}{\sqrt{[n\sum x^2 - (\sum x)^2][n\sum y^2 - (\sum y)^2]}}$ | 线性关系强度和方向 | |r| > 0.7 强 |
| 决定系数 r² | $1 - SS_{res}/SS_{total}$ | 模型解释的变异比例 | 越接近 1 越好 |
| 残差 e | $y - \hat{y}$ | 预测误差 | 随机分布 |

---

## ❌ 常见误区

### ❌ 误区 1: 相关就是因果
**✓ 正确理解**: 相关系数和回归分析只能揭示两个变量之间的线性关系，**不能证明因果关系**。可能存在第三个混淆变量同时影响 x 和 y。例如，学习时间与成绩相关，但可能是学习能力强的学生既学得久又考得好。

### ❌ 误区 2: 回归方程可以在任何范围内使用
**✓ 正确理解**: 回归方程**只在数据范围内有效**。超出数据范围的预测（外推）是不可靠的，因为我们不知道线性关系是否在范围外仍然成立。例如，用 2-12 小时的数据预测 20 小时的成绩是危险的。

### ❌ 误区 3: r² 高就说明模型好
**✓ 正确理解**: r² 高只说明自变量解释了因变量较大比例的变异，但**不保证模型假设成立**。必须检查残差的随机性、等方差性和正态性。r² 高但残差有模式说明模型可能有遗漏变量。

### ❌ 误区 4: 回归线上的所有点都是等权重的
**✓ 正确理解**: 远离 $\bar{x}$ 的数据点具有更高的**杠杆值**，对回归线的影响更大。强影响点可能不成比例地改变回归系数，需要通过 Cook 距离等方法识别。

### ❌ 误区 5: 残差之和不为零说明计算有误
**✓ 正确理解**: 由于**浮点数运算的舍入误差**，残差之和可能不精确为零，但应非常接近（如 10⁻¹⁰ 量级）。如果残差之和明显不为零，才需要检查计算过程。